In [1]:
# pp.ipynb - preprocess count data and save into formatted adata files.

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [3]:
feature_type = 'intergene'    # 'gene' or 'intergene'

In [4]:
seed_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/simu/matrix/HCC3N_simu.%s.overlap_uniq_filter.h5ad" % feature_type
cs_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/simu/matrix/HCC3N_simu.%s.sccnasim.h5ad" % feature_type

out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/analysis/sccnasim_cs_cna_signals/%s/pp" % feature_type

In [5]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [6]:
os.makedirs(out_dir, exist_ok = True)

### Seed data (scCNASim afc)

In [7]:
seed = ad.read_h5ad(seed_fn)
seed

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'

In [8]:
seed = seed[:, ~seed.var["feature"].duplicated(keep = False)].copy()
seed.obs.index = seed.obs["cell"]
seed.var.index = seed.var["feature"]
seed

AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'

In [9]:
seed.X.dtype

dtype('float64')

### scCNASim cs

In [10]:
cs = ad.read_h5ad(cs_fn)
cs

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 21215
    obs: 'cell_type', 'cell'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'

In [11]:
cs = cs[:, ~cs.var["feature"].duplicated(keep = False)].copy()
cs.obs.index = cs.obs["cell"]
cs.var.index = cs.var["feature"]
cs

AnnData object with n_obs × n_vars = 1200 × 21215
    obs: 'cell_type', 'cell'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'

In [12]:
cs.X.dtype

dtype('int32')

## Use intersect genes

In [13]:
adata_list = [seed, cs]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(21215,)

### Use the same order of genes

In [14]:
seed = seed[:, seed.var["feature"].isin(genes)].copy()
print(seed)

cs = cs[:, seed.var.index].copy()
print(cs)

AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'
AnnData object with n_obs × n_vars = 1200 × 21215
    obs: 'cell_type', 'cell'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'


## Split by cell type

In [15]:
seed_normal = seed[seed.obs["cell_type"] == normal_cell_type, :].copy()
print(seed_normal)

cs_normal = cs[cs.obs["cell_type"] == normal_cell_type, :].copy()
print(cs_normal)

cs_tumor = cs[cs.obs["cell_type"] == tumor_cell_type, :].copy()
print(cs_tumor)

AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'
AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell_type', 'cell'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'
AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell_type', 'cell'
    var: 'gene', 'chrom', 'start', 'end', 'cna_type', 'feature'


# Save Data

### Check the order of cells and genes

In [16]:
assert np.all(cs_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(cs_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())

In [17]:
seed_normal.var.index.name = None
cs_normal.var.index.name = None
cs_tumor.var.index.name = None

In [18]:
seed_normal.write_h5ad(os.path.join(out_dir, "seed_normal.h5ad"), compression = 'gzip')
cs_normal.write_h5ad(os.path.join(out_dir, "cs_normal.h5ad"), compression = 'gzip')
cs_tumor.write_h5ad(os.path.join(out_dir, "cs_tumor.h5ad"), compression = 'gzip')